#Demo aplicación modelo XGBoost

## 0. Setup

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import glob
from tqdm import tqdm
from google.colab import drive
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, cross_val_score, LearningCurveDisplay, StratifiedKFold, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
    recall_score,
    accuracy_score,
    recall_score,
    f1_score
)
import xgboost as xgb


In [2]:
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# Ruta de la carpeta
#ruta_base = "/content/drive/MyDrive/Reto_Telefonica/All_accents_csv/"
ruta_base = "/content/drive/MyDrive/00_Master/TFM/Reto_Telefonica/All_accents_csv/"

In [4]:
# Unificamos los CSVs para tener solo 1 dataframe con toda la información
archivos = glob.glob(ruta_base + "*.csv")

print("CSV encontrados:")
for a in archivos:
    print(a)

dfs = []

for archivo in archivos:

    # Leer CSV
    df = pd.read_csv(archivo)

    # Nombre del archivo
    nombre = os.path.basename(archivo).lower().strip()

    # LABEL

    if "real" in nombre:
        df["label"] = 0
    else:
        df["label"] = 1

    # ACCENT
    if "peruano" in nombre:
        df["accent"] = "peruano"

    elif "colombiano" in nombre:
        df["accent"] = "colombiano"

    elif "chileno" in nombre:
        df["accent"] = "chileno"

    elif "argentino" in nombre:
        df["accent"] = "argentino"

    else:
        df["accent"] = "desconocido"
        print("No se detectó acento en:", nombre)

    # GENERO
    if "genero_f" in df.columns:

        df["genero"] = np.where(
            df["genero_f"] == 1,
            "femenino",
            "masculino"
        )

    else:

        print("Warning: 'genero_f' no encontrada")
        df["genero"] = "unknown"

    # MODELO GENERADOR

    df["modelo_generador"] = "real"

    # Solo aplicar a sintéticos
    if df["label"].iloc[0] == 1:

        archivo_lower = df["archivo"].astype(str).str.lower()

        df.loc[
            archivo_lower.str.contains("tts-stargan", na=False),
            "modelo_generador"
        ] = "TTS-StarGAN"

        df.loc[
            archivo_lower.str.contains("tts-diff", na=False),
            "modelo_generador"
        ] = "TTS-Diff"

        df.loc[
            archivo_lower.str.contains("cyclegan", na=False),
            "modelo_generador"
        ] = "CycleGAN"

        df.loc[
            archivo_lower.str.contains("stargan", na=False)
            & ~archivo_lower.str.contains("tts-stargan", na=False),
            "modelo_generador"
        ] = "StarGAN"

        df.loc[
            archivo_lower.str.contains("diff", na=False)
            & ~archivo_lower.str.contains("tts-dif", na=False),
            "modelo_generador"
        ] = "Diff"

        df.loc[
            archivo_lower.str.contains("tts", na=False)
            & ~archivo_lower.str.contains("tts-stargan", na=False)
            & ~archivo_lower.str.contains("tts-dif", na=False),
            "modelo_generador"
        ] = "TTS"

    # Guardar dataframe
    dfs.append(df)

# UNIR TODO
df_unified = pd.concat(dfs, ignore_index=True)

print("\nShape final:")
print(df_unified.shape)

df_unified.head()



CSV encontrados:
/content/drive/MyDrive/00_Master/TFM/Reto_Telefonica/All_accents_csv/500peruano_dataset_features_sintetico.csv
/content/drive/MyDrive/00_Master/TFM/Reto_Telefonica/All_accents_csv/500peruano_dataset_features_real.csv
/content/drive/MyDrive/00_Master/TFM/Reto_Telefonica/All_accents_csv/500chileno_dataset_features_sintetico.csv
/content/drive/MyDrive/00_Master/TFM/Reto_Telefonica/All_accents_csv/500argentino_dataset_features_sintetico.csv
/content/drive/MyDrive/00_Master/TFM/Reto_Telefonica/All_accents_csv/500colombiano_dataset_features_sintetico.csv
/content/drive/MyDrive/00_Master/TFM/Reto_Telefonica/All_accents_csv/500colombiano_dataset_features_real.csv
/content/drive/MyDrive/00_Master/TFM/Reto_Telefonica/All_accents_csv/500argentino_dataset_features_real.csv
/content/drive/MyDrive/00_Master/TFM/Reto_Telefonica/All_accents_csv/500chileno_dataset_features_real.csv

Shape final:
(4000, 592)


,id_audio,archivo,label,genero_f,colombiano,chileno,argentino,peruano,modelo_cyclegan,modelo_diff,...,rolloff_median,rolloff_q1,rolloff_q3,rolloff_skew,rolloff_kurtosis,rolloff_mode,rolloff_iqr,accent,genero,modelo_generador
0,1,CycleGAN-pef_08421_00042078409-pef_01208_01247...,1,1,0,0,0,1,1,0,...,4761.71875,1335.937500,5916.015625,-0.121171,-1.659926,492.1875,4580.078125,peruano,femenino,CycleGAN
1,2,CycleGAN-pef_08421_00042078409-pef_03397_00805...,1,1,0,0,0,1,1,0,...,4093.75000,1810.546875,5324.218750,-0.099164,-1.290338,539.0625,3513.671875,peruano,femenino,CycleGAN
2,3,CycleGAN-pef_08421_00042078409-pef_07049_01353...,1,1,0,0,0,1,1,0,...,3882.81250,1744.140625,4939.453125,0.010095,-1.072997,515.6250,3195.312500,peruano,femenino,CycleGAN
3,4,CycleGAN-pef_08421_00042078409-pef_07508_01611...,1,1,0,0,0,1,1,0,...,4777.34375,2335.937500,5556.640625,-0.394409,-1.091107,531.2500,3220.703125,peruano,femenino,CycleGAN
4,5,CycleGAN-pef_08421_00569156593-pef_01208_01958...,1,1,0,0,0,1,1,0,...,2808.59375,1208.984375,5906.250000,0.294378,-1.419423,507.8125,4697.265625,peruano,femenino,CycleGAN


In [5]:
df_unified["genero"].unique()

array(['femenino', 'masculino'], dtype=object)

In [6]:
# DF excluyendo los peruanos
df_sin_peruano = df_unified[df_unified['peruano'] == 0].copy()

# DF solo peruano para probar al final
df_solo_peruano = df_unified[df_unified['peruano'] == 1].copy()

# Verificación de los tamaños
print(f"Audios para Train/Validation (sin acento peruano): {df_sin_peruano.shape[0]} filas")
print(f"Audios reservados para Test de Generalización (solo peruano): {df_solo_peruano.shape[0]} filas")

Audios para Train/Validation (sin acento peruano): 3000 filas
Audios reservados para Test de Generalización (solo peruano): 1000 filas


In [7]:
# Eliminamos columnas de metadatos para evitar sesgos
df_model = df_sin_peruano.copy()

columnas_a_eliminar = [
    "label",
    "archivo",
    "id_audio",
    "accent",
    "colombiano",
    "chileno",
    "argentino",
    "peruano",
    "modelo_cyclegan",
    "modelo_diff",
    "modelo_stargan",
    "modelo_tts_dif",
    "modelo_tts_stargan",
    "modelo_tts",
    "genero_f",
    "genero",
    "modelo_generador"
]

X = df_model.drop(columns=columnas_a_eliminar, errors="ignore")

y = df_model["label"]

# Verificar columnas restantes
print("Columnas restantes:")
print(X.columns)

# Ver primeras filas
print("\nPrimeras filas de X:")
display(X.head())

# Ver target
print("\nPrimeras filas de y:")
display(y.head())

Columnas restantes:
Index(['duracion_seg', 'zcr_mean', 'zcr_std', 'zcr_min', 'zcr_max',
       'zcr_median', 'zcr_q1', 'zcr_q3', 'zcr_skew', 'zcr_kurtosis',
       ...
       'rolloff_std', 'rolloff_min', 'rolloff_max', 'rolloff_median',
       'rolloff_q1', 'rolloff_q3', 'rolloff_skew', 'rolloff_kurtosis',
       'rolloff_mode', 'rolloff_iqr'],
      dtype='object', length=575)

Primeras filas de X:


,duracion_seg,zcr_mean,zcr_std,zcr_min,zcr_max,zcr_median,zcr_q1,zcr_q3,zcr_skew,zcr_kurtosis,...,rolloff_std,rolloff_min,rolloff_max,rolloff_median,rolloff_q1,rolloff_q3,rolloff_skew,rolloff_kurtosis,rolloff_mode,rolloff_iqr
1000,5.394938,0.118349,0.086926,0.008789,0.432617,0.089355,0.060547,0.149902,1.646221,2.615961,...,1737.794563,171.8750,7273.4375,3640.62500,1898.437500,4929.687500,-0.015848,-1.125790,2757.8125,3031.250000
1001,5.362938,0.129610,0.092042,0.008789,0.440430,0.100342,0.065918,0.165771,1.523157,2.178960,...,1919.135148,187.5000,7343.7500,4339.84375,2068.359375,5525.390625,-0.192997,-1.358087,1187.5000,3457.031250
1002,5.394938,0.127493,0.098236,0.006348,0.452637,0.101562,0.057129,0.166992,1.465031,1.951327,...,1952.102029,171.8750,7289.0625,4453.12500,2132.812500,5656.250000,-0.239998,-1.343159,2351.5625,3523.437500
1003,5.362938,0.127642,0.089768,0.010254,0.441895,0.098877,0.066895,0.159302,1.580699,2.347332,...,1761.820821,187.5000,7312.5000,4066.40625,2281.250000,5138.671875,-0.160017,-1.171392,1398.4375,2857.421875
1004,3.814938,0.121057,0.087096,0.006836,0.420410,0.095215,0.062256,0.166382,1.453465,2.087979,...,1921.594961,195.3125,7109.3750,3015.62500,1787.109375,5351.562500,0.170794,-1.353723,5351.5625,3564.453125



Primeras filas de y:


,label
1000,1
1001,1
1002,1
1003,1
1004,1


In [8]:
# TRAIN / TEST SPLIT
# Se divide el dataset en: entrenamiento (80%) y prueba (20%)
# stratify=y mantiene balanceadas las clases en ambos conjuntos

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("Shape Train:")
print(X_train.shape)

print("\nShape Test:")
print(X_test.shape)

Shape Train:
(2400, 575)

Shape Test:
(600, 575)


## Cargamos y entrenamos el modelo

In [9]:
#  Con la nueva parametrización, miramos que la recall siga siendo la misma

# 1. Definir el modelo
xgb_model = xgb.XGBClassifier(
    n_estimators=500,           # Suficientes árboles para aprender patrones complejos
    learning_rate=0.1,         # Podemos aumentarlo
    max_depth=6,                # Controla la complejidad de cada árbol
    subsample=0.7,              # Regularización: usa el 70% de las filas
    colsample_bytree=0.8,       # Regularización: usa el 80% de las columnas
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1
)


#2. Definir la estrategia de Cross-Validation
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# 3. Definir las métricas (Priorizando el Recall)
scoring_metrics = ['accuracy', 'recall', 'f1', 'precision']

# 4. Ejecutar Cross-Validation múltiple
cv_results = cross_validate(
    xgb_model,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring_metrics
)

# 5. Mostrar resultados
print("=== RESULTADOS DEL CROSS-VALIDATION EN TRAIN (5 Folds) ===")
print(f"RECALL promedio:    {np.mean(cv_results['test_recall']):.4f} (+/- {np.std(cv_results['test_recall']):.4f})")
print(f"F1-SCORE promedio:  {np.mean(cv_results['test_f1']):.4f} (+/- {np.std(cv_results['test_f1']):.4f})")
print(f"ACCURACY promedio:  {np.mean(cv_results['test_accuracy']):.4f} (+/- {np.std(cv_results['test_accuracy']):.4f})")

=== RESULTADOS DEL CROSS-VALIDATION EN TRAIN (5 Folds) ===
RECALL promedio:    0.9792 (+/- 0.0102)
F1-SCORE promedio:  0.9816 (+/- 0.0060)
ACCURACY promedio:  0.9817 (+/- 0.0060)


In [10]:
# 1. ENTRENAMIENTO: Entrenamos el modelo con TODO el conjunto de Train
xgb_model.fit(X_train, y_train)

# 2. PREDICCIÓN:
y_pred_xgb = xgb_model.predict(X_test)

## Funciones de extraer features del áudio

In [12]:
import gradio as gr
import pandas as pd
import numpy as np
import librosa
from scipy import stats
import os
# --- 1. FUNCIÓN DE CÁLCULO ESTADÍSTICO ---
def resumir_feature(vector, prefijo, debug=False):
    vector = np.asarray(vector).astype(float)

    if len(vector) == 0:
        return {}

    resultado = {
        f"{prefijo}_mean": np.mean(vector),
        f"{prefijo}_std": np.std(vector),
        f"{prefijo}_min": np.min(vector),
        f"{prefijo}_max": np.max(vector),
        f"{prefijo}_median": np.median(vector),
        f"{prefijo}_q1": np.quantile(vector, 0.25),
        f"{prefijo}_q3": np.quantile(vector, 0.75),
        f"{prefijo}_skew": stats.skew(vector),
        f"{prefijo}_kurtosis": stats.kurtosis(vector),
        f"{prefijo}_mode": stats.mode(vector, keepdims=True)[0][0],
        f"{prefijo}_iqr": stats.iqr(vector)
    }
    return resultado

# --- 2. FUNCIÓN DE EXTRACCIÓN PRINCIPAL ---
def extraer_features_produccion(ruta_audio):
    """
    Extrae features de un audio y devuelve un DataFrame de una fila listo para XGBoost.
    """
    # ⚠️ AJUSTA ESTOS VALORES A LOS QUE USASTE EN TU ENTRENAMIENTO
    sr_objetivo = 22050
    n_mfcc = 20

    y, sr = librosa.load(ruta_audio, sr=sr_objetivo)

    # Recorte de silencios
    y, _ = librosa.effects.trim(y)

    if len(y) < 512:
        raise ValueError("El audio es demasiado corto para ser analizado.")

    features = {}

    features["duracion_seg"] = len(y) / sr

    zcr = librosa.feature.zero_crossing_rate(y)[0]
    features.update(resumir_feature(zcr, "zcr"))

    rms = librosa.feature.rms(y=y)[0]
    features.update(resumir_feature(rms, "rms"))

    rmse_manual = np.sqrt(np.mean(y**2))
    features["rmse_manual"] = rmse_manual

    tempo = librosa.beat.tempo(y=y, sr=sr)[0]
    features["tempo"] = tempo

    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
    for i in range(n_mfcc):
        features.update(resumir_feature(mfcc[i], f"mfcc_{i+1}"))

    delta_mfcc = librosa.feature.delta(mfcc)
    for i in range(n_mfcc):
        features.update(resumir_feature(delta_mfcc[i], f"delta_mfcc_{i+1}"))

    delta2_mfcc = librosa.feature.delta(mfcc, order=2)
    for i in range(n_mfcc):
        features.update(resumir_feature(delta2_mfcc[i], f"delta2_mfcc_{i+1}"))

    centroid = librosa.feature.spectral_centroid(y=y, sr=sr)[0]
    features.update(resumir_feature(centroid, "centroid"))

    bandwidth = librosa.feature.spectral_bandwidth(y=y, sr=sr)[0]
    features.update(resumir_feature(bandwidth, "bandwidth"))

    contrast = librosa.feature.spectral_contrast(y=y, sr=sr)
    for i in range(contrast.shape[0]):
        features.update(resumir_feature(contrast[i], f"contrast_{i+1}"))

    flatness = librosa.feature.spectral_flatness(y=y)[0]
    features.update(resumir_feature(flatness, "flatness"))

    rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr)[0]
    features.update(resumir_feature(rolloff, "rolloff"))

    # Convertimos el diccionario final en un DataFrame de 1 fila
    return pd.DataFrame([features])

# --- 3. MOTOR DE PREDICCIÓN (GRADIO) ---
def predecir_audio(audio_path):
    if audio_path is None:
        return {"Esperando audio...": 1.0}

    try:
        # A) Extraemos las características
        df_features = extraer_features_produccion(audio_path)

        # B) Alineación de seguridad con el modelo XGBoost
        # Esto rellena con 0 cualquier variable que falte y asegura el orden exacto
        columnas_entrenamiento = xgb_model.feature_names_in_
        df_features = df_features.reindex(columns=columnas_entrenamiento, fill_value=0)

        # C) Predicción de Probabilidades
        probabilidades = xgb_model.predict_proba(df_features)[0]

        # ⚠️ IMPORTANTE: Verifica qué etiqueta es el 1 y el 0 en tu dataset.
        # Aquí asumo que 0 = Real, 1 = Fake. Si es al revés, invierte los índices [0] y [1].
        prob_real = float(probabilidades[0])
        prob_ia = float(probabilidades[1])

        return {"🗣️ Humano Real": prob_real, "🤖 Generado por IA": prob_ia}

    except Exception as e:
        return {f"Error: {str(e)}": 1.0}

## Preparamos demo

In [14]:
# --- 3. DISEÑO DE LA INTERFAZ WEB ---
demo = gr.Interface(
    fn=predecir_audio,
    inputs=gr.Audio(type="filepath", label="Sube o graba un audio para analizar"),
    outputs=gr.Label(num_top_classes=2, label="Veredicto del Modelo XGBoost"),
    title="🎙️ Detector de Deepfakes de Audio",
    description="Sube un archivo de voz. El algoritmo extraerá cientos de características acústicas (MFCCs, Energía, Contraste Espectral) y calculará mediante XGBoost la probabilidad de que el audio haya sido generado artificialmente.",
    theme="default"
)

## Launch demo

In [15]:
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://393ce6a3c4622f925b.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
